# 03. Data Preprocessing and Train/Test Split

Before fitting machine learning models, we must prepare the data carefully. This stage matters because data leakage is one of the most common mistakes in ML projects.

In this notebook we will:

- remove non-predictive columns
- encode the target variable as 0/1
- split the dataset into training and testing sets
- prepare for feature scaling where needed
- preserve the test set for honest evaluation

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
data_path = project_root / 'data' / 'data.csv'

df = pd.read_csv(data_path)
df = df.drop(columns=[col for col in df.columns if 'Unnamed:' in str(col)], errors='ignore')
df = df.drop(columns=['id'], errors='ignore')

df['diagnosis'] = df['diagnosis'].astype(str).str.strip()
print(df.shape)
print(df.head())

In [ ]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis'].map({'B': 0, 'M': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train distribution:', y_train.value_counts().to_dict())
print('y_test distribution:', y_test.value_counts().to_dict())

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000, random_state=42)),
])

model.fit(X_train, y_train)
print('Train score:', model.score(X_train, y_train))
print('Test score:', model.score(X_test, y_test))

## Why scaling matters

Some models, such as logistic regression and SVM, are sensitive to feature scale because the optimization depends on the magnitude of the coefficients and distances. Standardization keeps these features on a similar numerical scale and can improve optimization stability.

The train/test split happens before any learning-based transformation. This is important because scaling should be fitted only on the training data; otherwise, information from the test set leaks into the model.